In [3]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter, KaplanMeierFitter
from sklearn.linear_model import LogisticRegression
import warnings
warnings.simplefilter(action='ignore')

In [4]:
data_cox = pd.read_csv('synthetic_data.csv')
print('Number of patients: %d' % data_cox.shape[0])
cols_basic = ['duration', 'event', 'fmi_test'] # Basic infomation for survival analysis
confounders = ['Female', 'Age >= 40', 'ECOG >= 3'] # Confounders
data_cox.head()

Number of patients: 3000


,Female,Age >= 40,ECOG >= 3,Gene A,Gene B,Treatment 1,Treatment 2,duration,event,fmi_test
0,1,1,0,0,0,0,1,439,1,31
1,1,1,1,1,0,1,0,258,0,9
2,1,1,1,1,1,0,1,132,1,31
3,0,0,0,0,0,0,1,62,1,28
4,0,0,0,0,0,0,1,46,1,32


In [5]:
def process_HR(cph_summary, name_feature):
    '''
    Process result of CoxPH model to HR with 95% confidence intervals.
    '''
    result = cph_summary.loc[name_feature, ['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%']].values
    HR = '%.2f (%.2f, %.2f)' % tuple(result)   
    return HR
    
def Cox_IPTW(data_fit, name_feature):
    '''
    Uni-covariate CoxPH model with IPTW and left-truncation. 
    name_feature is the feature we are interested in (e.g., gene name or treatment name).
    Return the HR for name_feature.
    '''
    # Generate data used in Cox Regression with IPTW Weights
    data_confounders = data_fit.iloc[:, 3:].copy().drop(columns=[name_feature])
    df = pd.concat([data_fit.iloc[:, :3], data_fit[[name_feature]]], axis=1)
    X = data_confounders.values
    y = data_fit[name_feature].values
    model = LogisticRegression(solver='lbfgs', n_jobs=-1, class_weight='balanced')
    model.fit(X, y)
    p_treated = float(np.sum(y==1))/y.shape[0]
    propensity_scores = model.predict_proba(X)[:, 1]
    df['weights'] = 0
    IP_treated = 1 / propensity_scores
    IP_untreated = 1 / (1 - propensity_scores)        
    df.loc[df[name_feature]==1, 'weights'] = IP_treated[df[name_feature]==1]
    df.loc[df[name_feature]==0, 'weights'] = IP_untreated[df[name_feature]==0]    
    # Cox Regression
    cph = CoxPHFitter()
    cph.fit(df, 'duration', 'event', weights_col='weights', entry_col='fmi_test', robust=True)
    HR = process_HR(cph.summary, name_feature)
    return HR

In [6]:

data_fit = data_cox[cols_basic+confounders+['Gene A']]
HR = Cox_IPTW(data_fit, 'Gene A')
print('OS HR for Gene A: %s' % (HR))

OS HR for Gene A: 1.09 (1.01, 1.19)


In [7]:
gene = 'Gene A'
treatment = 'Treatment 1'

In [8]:
data_fit = data_cox[cols_basic+[gene, treatment]]
col_inter = '%s+%s'%(gene, treatment)
data_fit[col_inter] = data_fit[gene]*data_fit[treatment]
cph = CoxPHFitter()
cph.fit(data_fit, 'duration', 'event', entry_col='fmi_test', robust=True)
HR = process_HR(cph.summary, col_inter)
print('Interaction between Gene A and Treatment 1 is %s' % HR)

Interaction between Gene A and Treatment 1 is 0.65 (0.55, 0.77)


In [9]:
data_fit = data_cox.loc[data_cox[gene]==1, cols_basic+confounders+[treatment]].copy()
HR = Cox_IPTW(data_fit, treatment)
print('Treatment HR for patients with mutations of Gene A = %s' % HR)

data_fit = data_cox.loc[data_cox[gene]==0, cols_basic+confounders+[treatment]].copy()
HR = Cox_IPTW(data_fit, treatment)
print('Treatment HR for patients without mutations of Gene A = %s' % HR)

Treatment HR for patients with mutations of Gene A = 0.79 (0.69, 0.89)
Treatment HR for patients without mutations of Gene A = 1.20 (1.08, 1.34)


In [10]:
data_fit = data_cox.loc[data_cox[treatment]==1, cols_basic+confounders+[gene]].copy()
HR = Cox_IPTW(data_fit, gene)
print('Gene A mutation HR for patients who received Treatment 1 = %s' % HR)

data_fit = data_cox.loc[data_cox[treatment]==0, cols_basic+confounders+[gene]].copy()
HR = Cox_IPTW(data_fit, gene)
print('Gene A mutation HR for patients who didn\'t receive Treatment 1 = %s' % HR)

Gene A mutation HR for patients who received Treatment 1 = 0.90 (0.81, 1.01)
Gene A mutation HR for patients who didn't receive Treatment 1 = 1.41 (1.24, 1.60)


In [11]:
gene_anchor = 'Gene A'
gene_modifier = 'Gene B' 
data_fit = data_cox.loc[data_cox[gene_anchor]==1, cols_basic+confounders+[gene_modifier]]
HR = Cox_IPTW(data_fit, gene_modifier)
print('modifier gene HR for anchor gene positive: %s' % HR)

modifier gene HR for anchor gene positive: 0.85 (0.75, 0.97)
